In [ ]:
"""Wasserstein risk index (distribution shift) for futures.

This notebook demonstrates a small end-to-end research pipeline:

- Load continuous daily futures prices (placeholder fake generator for now)
- Compute daily log returns
- Build the Wasserstein distribution-shift index W_t from cross-sectional returns
- Build a core panel with realized vol + rolling largest eigenvalue
- Run a simple HAC regression, an event study, and a conditioning experiment

TODO: Replace the fake loader in `algogators_wrisk.data.load_continuous_futures_prices`
with your internal `algogators-data` call.
"""

## Setup & imports

This notebook is a thin front-end: it calls into `algogators_wrisk` for all computations.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make repo root importable when running from notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from algogators_wrisk import config
from algogators_wrisk import data, features, analysis

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

np.random.seed(0)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

In [ ]:
# Research parameters (override defaults here if desired)
universe = config.UNIVERSE
start_date = config.START_DATE
end_date = config.END_DATE

rv_past_window = config.RV_PAST_WINDOW
rv_future_window = config.RV_FUTURE_WINDOW
lambda1_window = config.LAMBDA1_WINDOW

w_event_q = config.W_EVENT_QUANTILE
exposure_on_event = config.EXPOSURE_ON_EVENT
hac_lags = config.HAC_LAGS

print(f"Universe size: {len(universe)}")
print(f"Date range: {start_date} → {end_date}")
print(f"RV windows (past/future): {rv_past_window}/{rv_future_window}")
print(f"Lambda1 window: {lambda1_window}")
print(f"Event quantile: {w_event_q}")
print(f"Exposure on event: {exposure_on_event}")
print(f"HAC lags: {hac_lags}")

## Load data

We use the placeholder loader in `algogators_wrisk.data` (fake, deterministic) so the notebook runs end-to-end.

(parameters are set in the code cell above)

## Compute returns and core panel

Compute log returns, build the cross-sectional return matrix, then build the core daily panel used for regression / event studies / strategy conditioning.

In [ ]:
# 1) Load continuous futures prices (FAKE placeholder)
prices = data.load_continuous_futures_prices(
    universe=universe,
    start_date=start_date,
    end_date=end_date,
    seed=42,
)

display(prices.head())
print(prices.shape)

In [ ]:
# 2) Compute daily log returns
returns = data.compute_log_returns(prices)
ret_matrix = features.build_return_matrix(returns, universe=universe)

display(ret_matrix.head())
print(ret_matrix.shape)

## Basic diagnostics

Plot the distribution-shift index \(W_t\), realized volatility, and the rolling largest correlation eigenvalue.

In [ ]:
# 3) Compute W_t, realized volatility, and rolling lambda1 via a single panel
panel = analysis.build_core_panel(
    ret_matrix,
    rv_past_window=rv_past_window,
    rv_future_window=rv_future_window,
    lambda1_window=lambda1_window,
    annualize_rv=True,
)

display(panel.tail())
print(panel.isna().mean().sort_values(ascending=False).head(10))

## Basic diagnostics (plots)

Plot \(W_t\), realized volatility, and \(\lambda_1\) over time.

In [ ]:
# Plot W_t, realized vol, and lambda1
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

panel["W"].plot(ax=axes[0], color="black", lw=1)
axes[0].set_title("Wasserstein distribution-shift index (W)")

panel[["rv_past", "rv_future"]].plot(ax=axes[1], lw=1)
axes[1].set_title("Realized volatility (past vs future)")

panel["lambda1"].plot(ax=axes[2], color="tab:purple", lw=1)
axes[2].set_title("Rolling largest correlation eigenvalue (lambda1)")

plt.tight_layout()
plt.show()

In [ ]:
# 5) Regression: rv_future ~ W + rv_past + lambda1 (HAC SEs)
res = analysis.run_rv_regression(panel, hac_lags=hac_lags)
print(res.summary())

## Event study

Event days are defined as high-\(W_t\) days (quantile threshold). We plot the average path of `mkt_ret` around events.

In [ ]:
es = analysis.make_event_study_dataset(
    panel,
    value_col="mkt_ret",
    quantile=w_event_q,
    pre=10,
    post=10,
    min_gap=5,
)

print(f"Identified {len(es.events)} event days (after de-clustering).")
display(es.stacked.head())

plt.figure(figsize=(10, 4))
es.avg_path.plot(marker="o")
plt.axvline(0, color="black", lw=1)
plt.title("Event study: average mkt_ret around high-W events")
plt.xlabel("tau (days from event)")
plt.ylabel("avg return")
plt.tight_layout()
plt.show()

## Strategy conditioning

Compare a baseline exposure strategy vs. scaling exposure down on high-\(W_t\) days. Compute basic stats: mean, stdev, Sharpe, max drawdown.

In [ ]:
# Baseline vs conditioned exposure (scaled down on high-W days)
strat = analysis.run_strategy_conditioning_experiment(
    panel,
    quantile=w_event_q,
    exposure_on_event=exposure_on_event,
)

display(strat.head())

# --- performance helpers ---
def sharpe_ratio(x: pd.Series, *, periods_per_year: int = 252) -> float:
    x = x.dropna()
    if x.std() == 0 or len(x) < 2:
        return np.nan
    return float(np.sqrt(periods_per_year) * x.mean() / x.std())


def max_drawdown(cum_pnl: pd.Series) -> float:
    """Max drawdown on cumulative PnL series (additive)."""
    c = cum_pnl.dropna()
    if c.empty:
        return np.nan
    running_max = c.cummax()
    dd = c - running_max
    return float(dd.min())


stats = pd.DataFrame(
    {
        "mean": [strat["baseline_pnl"].mean(), strat["conditioned_pnl"].mean()],
        "stdev": [strat["baseline_pnl"].std(), strat["conditioned_pnl"].std()],
        "sharpe": [
            sharpe_ratio(strat["baseline_pnl"]),
            sharpe_ratio(strat["conditioned_pnl"]),
        ],
        "max_dd": [
            max_drawdown(strat["baseline_cum"]),
            max_drawdown(strat["conditioned_cum"]),
        ],
    },
    index=["baseline", "conditioned"],
)

print(f"Event rate: {strat['is_event'].mean():.3%}")
display(stats)

plt.figure(figsize=(12, 4))
strat[["baseline_cum", "conditioned_cum"]].plot(lw=1.5)
plt.title("Cumulative PnL (baseline vs conditioned)")
plt.xlabel("date")
plt.ylabel("cum pnl (sum of daily returns)")
plt.tight_layout()
plt.show()

## Notes

- The price loader is currently **fake** and deterministic.
- Once you wire real prices, everything downstream should work unchanged.

In [ ]:
# Normalize the features
X_normalized = features.normalize_features(df_returns)
X_normalized_df = pd.DataFrame(
    X_normalized,
    columns=[f'{col}_norm' for col in df_returns.columns]
)

print("Normalized Features Statistics:")
print(X_normalized_df.describe())

# Scale features to [0, 1] range
X_scaled = features.scale_features(df_returns)
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=[f'{col}_scaled' for col in df_returns.columns]
)

print("\nScaled Features Statistics:")
print(X_scaled_df.describe())

In [ ]:
print("Configuration Settings:")
print(f"Data Path: {config.DATA_PATH}")
print(f"Output Path: {config.OUTPUT_PATH}")
print(f"Random Seed: {config.RANDOM_SEED}")
print(f"Test Size: {config.TEST_SIZE}")
print(f"Default Wasserstein P: {config.DEFAULT_WASSERSTEIN_P}")

# Set random seed for reproducibility
np.random.seed(config.RANDOM_SEED)